# UTD19 → ML → SUMO : full end-to-end run on ColabReimplementation of **"Advanced Traffic Demand Generation in SUMO: ML-based Prediction ofFlow Rate based on Real-world Measured Datasets"** (Qiu, Alemada Sathish, Wang, Schotten,IEEE VTC Spring 2024).### What the paper actually doesIt is **not** "SUMO data → ML". It is the opposite direction:```Real UTD19 measurements (40 cities, 23 577 detectors, 169 032 955 rows)        │  train Linear / SVM / Polynomial / Decision Tree / Random Forest        ▼Predicted traffic flow  (vehicles per hour, per road, per hour-of-day)        │  written as SUMO edge counts        ▼routeSampler.py picks a subset of candidate routes so that thevehicles crossing each monitored edge match the predicted counts        ▼24-hour SUMO simulation of Berlin        ▼Compare simulated counts vs predicted (GEH) and vs real Berlin counts```The ML model supplies the **demand**; SUMO supplies the **traffic**.### The 21 pipeline steps| Steps | Paper section | What happens ||---|---|---|| 01–02 | III-A | Download 6.98 GB UTD19, aggregate all 169 M rows to hourly || 03 | V-A / Table V | Train 5 models, compare SMAPE + MAE || 04–05 | III-B / Table III | Berlin open-data baseline (Feb 2021) || 06–07 | IV-B | Berlin OSM → `netconvert` → SUMO network || 08 | IV-B | Match the 216 detectors onto network edges || 09–10 | IV-A / Table IV | Predict Berlin flow for every detector-hour || 11–13 | IV-C | randomTrips + TAZ trips + duarouter candidate routes || 14–15 | IV-D / Fig. 3 | Predicted counts → `routeSampler.py` calibration || 16–17 | V-C | 24-hour SUMO run || 18–20 | V-B, V-C / Fig. 4, 5 | GEH, actual-vs-predicted, packaging |**Runtime ≈ 3 hours.** Use `Runtime → Change runtime type → High-RAM`.Every step checkpoints, so if Colab disconnects, re-run the same cell and it resumes.

## 1. Check the machineYou want High-RAM. The 6.98 GB download plus DuckDB spill needs ~40 GB of free disk.

In [ ]:
import multiprocessing, shutil, subprocess, sysprint("Python :", sys.version.split()[0])print("CPUs   :", multiprocessing.cpu_count())with open("/proc/meminfo") as f:    total_kb = int(f.readline().split()[1])print(f"RAM    : {total_kb/1024/1024:.1f} GB")free = shutil.disk_usage("/content").free / 1e9print(f"Disk   : {free:.0f} GB free on /content")if total_kb/1024/1024 < 20:    print("\n[!] Standard runtime detected. Runtime > Change runtime type > High-RAM"          "\n    is strongly recommended, or the tree models may be OOM-killed.")

## 2. Mount Drive`/content` is fast but wiped when the session ends. Drive is slow but permanent, so thebig dataset stays on `/content` and only the **results** are copied to Drive at the end.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import os# Your shared Drive folder (id 1N6VjBmcrs131360PbCaidqb3LzOfg55V). If Drive shows# it under a different name, set RESULTS_DIR to whatever path appears in the# file browser on the left, then re-run this cell.RESULTS_DIR = '/content/drive/MyDrive/thesis_results'os.makedirs(RESULTS_DIR, exist_ok=True)print("Results will be archived to:", RESULTS_DIR)# Show the top level of your Drive so you can spot the right folder.for name in sorted(os.listdir('/content/drive/MyDrive'))[:40]:    print("  ", name)

## 3. Clone the private repositoryCreate a token at **github.com → Settings → Developer settings → Personal access tokens →Fine-grained tokens**, give it *Contents: Read* on this repo, and paste it when asked.It is read with `getpass`, so it never appears in the notebook output.

In [ ]:
import os, subprocessfrom getpass import getpassGITHUB_USER = "auth889-ai"REPO        = "thesis"BRANCH      = "main"PROJECT     = f"/content/{REPO}/utd19_berlin_sumo_e2e_v3"if not os.path.exists(f"/content/{REPO}"):    token = getpass("GitHub token: ")    url = f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"    subprocess.run(["git", "clone", "--branch", BRANCH, url, f"/content/{REPO}"], check=True)    # Drop the token from the stored remote so it is not left on disk.    subprocess.run(["git", "-C", f"/content/{REPO}", "remote", "set-url", "origin",                    f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=True)else:    print("Already cloned. To pull the newest code, run the cell below.")os.chdir(PROJECT)print("Working directory:", os.getcwd())

In [ ]:
# Re-run this after every `git push` from VS Code to pick up your latest edits.from getpass import getpassimport subprocesstoken = getpass("GitHub token (for pull): ")subprocess.run(["git", "-C", "/content/thesis", "pull",                f"https://{token}@github.com/auth889-ai/thesis.git", "main"], check=True)

## 4. Install SUMO and the Python stack`eclipse-sumo` ships the real `sumo`, `netconvert` and `duarouter` binaries as a wheel,so no apt/PPA is needed. `sumolib` and `traci` are separate packages that severalpipeline modules import directly.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y osmium-tool > /dev/null!pip install -q -r requirements-colab.txtprint("install finished")

In [ ]:
# Verify the real binaries and that the pipeline can locate SUMO_HOME.import subprocess, syssys.path.insert(0, '/content/thesis/utd19_berlin_sumo_e2e_v3')from src.utils import find_sumo_home, sumo_toolfor exe in ["sumo", "netconvert", "duarouter", "osmium"]:    v = subprocess.run([exe, "--version"], capture_output=True, text=True).stdout.splitlines()    print(f"{exe:12s} {v[0] if v else 'NOT FOUND'}")home = find_sumo_home()print("\nSUMO_HOME  :", home)print("randomTrips:", sumo_tool("randomTrips.py").exists())print("routeSampler:", sumo_tool("routeSampler.py").exists())

## 5. The configuration`colab_3h.yaml` streams **every** published UTD19 row in step 02. Only the model *fitting*uses a reproducible reservoir sample, because a Random Forest over ~10 M hourly rows needsfar more RAM than any Colab VM. All caps are recorded in `training_protocol.json`.

In [ ]:
CONFIG = "config/colab_3h.yaml"print(open(CONFIG).read())

In [ ]:
import subprocess, time, sysdef run_steps(from_step, to_step, config=CONFIG):    "Run part of the pipeline, streaming output live."    cmd = [sys.executable, "-m", "src.pipeline", "--config", config,           "--from-step", from_step, "--to-step", to_step,           "--accept-auto-mapping"]    print("$", " ".join(cmd), flush=True)    start = time.time()    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,                         text=True, bufsize=1)    for line in p.stdout:        print(line, end="")    p.wait()    mins = (time.time() - start) / 60    if p.returncode != 0:        raise RuntimeError(f"pipeline failed in {from_step}..{to_step} after {mins:.1f} min")    print(f"\n=== {from_step} .. {to_step} finished in {mins:.1f} min ===")!python -m src.pipeline --config $CONFIG --list-steps

## 6. Steps 01–02 — the full UTD19 dataset  ⏱ ~35–60 minDownloads `utd19_u.csv` (6.98 GB, 169 032 955 rows) from the ETH Research Collection,then aggregates it with DuckDB.Two things the paper leaves implicit, both handled here:* UTD19 `flow` is already **veh/h/lane**, even though intervals are 3–5 min. So the hourly  value is the *mean of the interval rates*, not a sum.* The manual defines `error` as *"identified or reported error if a non-missing value is  reported"* — so rows with **any** non-null `error` are dropped.The target is `flow × monitored lanes`, to be comparable with Berlin's per-cross-section`q_kfz` counts.

In [ ]:
run_steps("01_download_utd19", "02_preprocess_utd19")

In [ ]:
import json, pandas as pdWS = "/content/utd19_berlin_e2e"audit = json.load(open(f"{WS}/processed/utd19/utd19_audit.json"))print(json.dumps(audit, indent=2))cov = pd.read_csv(f"{WS}/processed/utd19/city_coverage.csv")print(f"\n{len(cov)} cities")display(cov.sort_values("rows", ascending=False).head(15))

## 7. Step 03 — train the models  ⏱ ~20–30 minThe paper compares Linear Regression, Linear SVM, Polynomial Regression, Decision Trees andRandom Forest by **SMAPE** and **MAE** (Table V), and picks Random Forest.The split is a **group holdout by whole detector** — no detector appears in both train andtest — so the score measures generalisation to unseen roads, which is what the Berlintransfer actually requires.

In [ ]:
run_steps("03_train_models", "03_train_models")

In [ ]:
import pandas as pd, json# Our metrics, ranked.metrics = pd.read_csv(f"{WS}/models/model_comparison.csv")display(metrics)# Side by side with the paper's Table V (written by the pipeline itself).display(pd.read_csv(f"{WS}/models/paper_vs_reimplementation.csv"))print("Chosen for Berlin:",      json.load(open(f"{WS}/models/training_protocol.json")).get("selected_model"))print("\nTable V reproduction: what must match is the ORDERING - Random Forest best,")print("linear models worst. Absolute values differ because the authors published")print("neither their split, their row filter, nor their hyperparameters, so")print("exact_match_expected is False by construction.")

## 8. Steps 04–05 — the Berlin baseline  ⏱ ~2 minBerlin's hourly `Messquerschnitte` files (paper Table III) for February 2021, restricted tothe five weekdays plotted in Figure 4. `Stammdaten_Verkehrsdetektion.xlsx` maps each`MQ_KURZNAME` (`TE001`) to WGS84 coordinates and a monitored-lane count.

In [ ]:
run_steps("04_download_berlin", "05_preprocess_berlin")

In [ ]:
import json, pandas as pdprint(json.dumps(json.load(open(f"{WS}/processed/berlin/berlin_audit.json")), indent=2))display(pd.read_parquet(f"{WS}/processed/berlin/berlin_actual.parquet").head())

## 9. Steps 06–08 — Berlin network + detector mapping  ⏱ ~25–40 min`netconvert` turns the Geofabrik Berlin extract into a SUMO network (this is the slow one).Then each detector is matched to a network edge by distance, road-name similarity andheading agreement. `mapping_review.html` lets you audit the matches afterwards.

In [ ]:
run_steps("06_download_osm", "08_map_detectors")

In [ ]:
import json, pandas as pdprint(json.dumps(json.load(open(f"{WS}/sumo/network/network_stats.json")), indent=2))m = pd.read_csv(f"{WS}/sumo/mapping/mapping_accepted.csv")print(f"\n{len(m)} detectors mapped onto edges")display(m.head())

## 10. Steps 09–10 — predict Berlin flow  ⏱ ~2 minBuild a feature row for every (detector, hour) using the Berlin road attributes, then applythe trained Random Forest. This is the paper's Table IV: one predicted flow per hour.

In [ ]:
run_steps("09_build_berlin_features", "10_predict_berlin")

In [ ]:
import pandas as pdpred = pd.read_csv(f"{WS}/processed/berlin/berlin_predictions.csv")print(pred.shape)display(pred.head(10))# Paper Table IV shows hours 00:00-05:00 for one detectorone = pred[pred.detid == pred.detid.iloc[0]].sort_values("hour")print(f"\nDetector {one.detid.iloc[0]} predicted flow by hour (cf. Table IV):")print(one[["hour", "predicted_flow"]].head(6).to_string(index=False))

## 11. Steps 11–15 — candidate routes and calibration  ⏱ ~45–70 minThis is Figure 3 of the paper. `routeSampler.py` cannot invent routes — it can only *choose*from routes you give it. So first we generate candidates three ways:1. `randomTrips.py` → `duarouter` — general background traffic2. **TAZ trips** — so every zone of the city is covered3. **coverage trips** — targeted routes forced through each monitored edge, because   randomly generated routes will not reliably cross all 216 detector edgesThen the predicted counts become a SUMO `edgeData` file and `routeSampler --optimize full`solves the linear program that picks how often each route is used so the edge counts match.

In [ ]:
run_steps("11_create_taz", "15_route_sampler")

In [ ]:
import jsonprint(json.dumps(json.load(open(f"{WS}/sumo/candidates/candidate_coverage_summary.json")), indent=2)[:1500])print(json.dumps(json.load(open(f"{WS}/sumo/calibration/route_sampler_summary.json")), indent=2))

## 12. Steps 16–17 — the 24-hour Berlin simulation  ⏱ ~30–60 minThe real SUMO engine, 86 400 simulated seconds, writing per-edge hourly counts plus anetwork-wide summary.

In [ ]:
run_steps("16_build_simulation", "17_run_simulation")

## 13. Steps 18–20 — results  ⏱ ~5 minGEH per detector-hour (the paper's acceptance criterion is **GEH < 5 for 85 % of detectors**),predicted-vs-actual error, and the network trajectory of Figure 5.

In [ ]:
run_steps("18_extract_outputs", "20_package_gui")

In [ ]:
import jsonev = json.load(open(f"{WS}/reports/final_evaluation.json"))print(json.dumps({k: v for k, v in ev.items() if k != "simulation_extraction"}, indent=2))geh = ev["GEH"]print("\n" + "="*60)print(f"GEH < 5 : {geh['GEH_lt_5_percent']:.1f}%   (paper claims > 85%)")print(f"vehicles: {ev['simulation_extraction']['inserted_vehicles']:,}   (paper ~1.2 M)")print("="*60)

## 14. Reproduce Figure 4 and Figure 5

In [ ]:
import matplotlib.pyplot as plt, pandas as pdpred = pd.read_csv(f"{WS}/processed/berlin/berlin_predictions.csv")actual = pd.read_parquet(f"{WS}/processed/berlin/berlin_actual.parquet")# ---- Figure 4: predicted vs actual for one detector, over the day ----det = pred.detid.iloc[0]fig, ax = plt.subplots(figsize=(9, 5))a = actual[actual.detid == det]for date, grp in a.groupby(a["date"].astype(str)):    ax.plot(grp["hour"], grp["actual_flow"], lw=1.2, label=f"actual {date}")p = pred[pred.detid == det].sort_values("hour")ax.plot(p["hour"], p["predicted_flow"], "b--", lw=2.5, label="predicted")ax.set_xlabel("interval (hours)"); ax.set_ylabel("Traffic count")ax.set_title(f"Figure 4 reproduction - detector {det}")ax.legend(fontsize=8); ax.grid(alpha=.3)plt.tight_layout(); plt.savefig(f"{WS}/figures/figure4.png", dpi=150); plt.show()

In [ ]:
# ---- Figure 5: vehicles running in the simulation across 24 h ----import xml.etree.ElementTree as ET, matplotlib.pyplot as plttimes, running = [], []for _, el in ET.iterparse(f"{WS}/sumo/simulation/summary.xml", events=("end",)):    if el.tag == "step":        times.append(float(el.get("time")) / 3600)        running.append(int(el.get("running")))        el.clear()fig, ax = plt.subplots(figsize=(9, 5))ax.plot(times, running, lw=1.5, label="running in simulation")ax.set_xlabel("Time (h)"); ax.set_ylabel("Number of cars running in simulation")ax.set_title("Figure 5 reproduction - 24 h Berlin trajectory")ax.legend(); ax.grid(alpha=.3)plt.tight_layout(); plt.savefig(f"{WS}/figures/figure5.png", dpi=150); plt.show()print(f"peak concurrent vehicles: {max(running):,}")

## 15. Save everything to Drive`/content` disappears when the session ends — this is the step that makes the run permanent.

In [ ]:
!python -m src.archive_workspace --workspace $WS --destination "$RESULTS_DIR/utd19_berlin_run"!ls -lh "$RESULTS_DIR"

In [ ]:
# Also copy the small stuff uncompressed, so it is browsable in Drive.import shutil, osfor sub in ["reports", "figures", "models"]:    dst = f"{RESULTS_DIR}/{sub}"    shutil.rmtree(dst, ignore_errors=True)    shutil.copytree(f"{WS}/{sub}", dst,                    ignore=shutil.ignore_patterns("*.joblib"))print(os.listdir(RESULTS_DIR))

## 16. Watch it in the SUMO GUI (on your Mac)Colab cannot show the SUMO GUI. `gui_bundle.zip` in Drive contains the network, thecalibrated routes and a ready `.sumocfg`. On your Mac:```bashpip install eclipse-sumo sumolib traciunzip gui_bundle.zip -d berlin_gui && cd berlin_guisumo-gui -c berlin.sumocfg```Press **Run**. That is the paper's Berlin scenario, driven by the flows your Random Forestpredicted.---## If the session diesNothing is lost. Re-run the setup cells (1–5), then re-run whichever `run_steps(...)` cellyou were on — completed steps are skipped via their checkpoint files in`/content/utd19_berlin_e2e/checkpoints/`.To force a step to re-run, delete its checkpoint:```python!rm /content/utd19_berlin_e2e/checkpoints/15_route_sampler.done.json```## What you may and may not claim**Yes:** a paper-faithful end-to-end reconstruction on the official public data, using thesame SUMO tools, with independently generated results.**No:** exact reproduction of Table V or of the 1.2 M-vehicle scenario. The authors did notpublish their code, their row filter, their split, their hyperparameters, their OSMsnapshot, their TAZ definitions or their candidate-route counts. See`docs/REPRODUCIBILITY_GAPS.md`.